# Malos Controles en Python
## Econometría Avanzada - Ana María Díaz

Este notebook demuestra por qué controlar ciertas variables puede sesgar los resultados.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

np.random.seed(2468)

## PARTE 1: Mediador Puro
### DGP: D → M → Y

Si el objetivo es el **efecto total** de D en Y, **no debemos controlar M**.

In [ ]:
n = 10000
a = 2  # D -> M
b = 1  # M -> Y

D = np.random.binomial(1, 0.5, n)
M = a * D + np.random.normal(0, 1, n)
Y = b * M + np.random.normal(0, 1, n)

# Regresión sin mediador (efecto TOTAL)
print("=== Regresión sin mediador (efecto TOTAL) ===")
X_total = sm.add_constant(D)
model_total = sm.OLS(Y, X_total).fit()
print(f"Coef D: {model_total.params[1]:.4f} (esperado ~{a*b})")

# Regresión con mediador (efecto DIRECTO)
print("\n=== Regresión con mediador (MAL CONTROL) ===")
X_directo = sm.add_constant(np.column_stack([D, M]))
model_directo = sm.OLS(Y, X_directo).fit()
print(f"Coef D: {model_directo.params[1]:.4f} (esperado ~0)")

## PARTE 2: Colisionador Clásico
### DGP: D → C ← Y (D y Y independientes)

Controlar C **abre** un camino espurio entre D y Y.

In [ ]:
n = 1000
D = np.random.normal(0, 1, n)
Y = np.random.normal(0, 1, n)  # D y Y independientes!
C = 2*D - 0.5*Y + np.random.normal(0, 1, n)  # Colisionador

# Sin colisionador (CORRECTO)
print("=== Sin colisionador (CORRECTO) ===")
X_noC = sm.add_constant(D)
model_noC = sm.OLS(Y, X_noC).fit()
print(f"Coef D: {model_noC.params[1]:.4f} (esperado ~0)")

# Con colisionador (MAL CONTROL)
print("\n=== Con colisionador (MAL CONTROL) ===")
X_conC = sm.add_constant(np.column_stack([D, C]))
model_conC = sm.OLS(Y, X_conC).fit()
print(f"Coef D: {model_conC.params[1]:.4f} (SESGADO!)")

## PARTE 3: Monte Carlo - Mediador

In [ ]:
n_reps = 300
b_total = []
b_directo = []

for _ in range(n_reps):
    D = np.random.binomial(1, 0.5, 3000)
    M = 2 * D + np.random.normal(0, 1, 3000)
    Y = 1 * M + np.random.normal(0, 1, 3000)
    
    # Total
    X_t = sm.add_constant(D)
    b_total.append(sm.OLS(Y, X_t).fit().params[1])
    
    # Directo
    X_d = sm.add_constant(np.column_stack([D, M]))
    b_directo.append(sm.OLS(Y, X_d).fit().params[1])

print(f"Promedio b_total: {np.mean(b_total):.4f} (esperado ~2)")
print(f"Promedio b_directo: {np.mean(b_directo):.4f} (esperado ~0)")

## PARTE 4: Monte Carlo - Colisionador

In [ ]:
b_noC = []
b_conC = []

for _ in range(n_reps):
    D = np.random.normal(0, 1, 1000)
    Y = np.random.normal(0, 1, 1000)
    C = 2*D - 0.5*Y + np.random.normal(0, 1, 1000)
    
    X_n = sm.add_constant(D)
    b_noC.append(sm.OLS(Y, X_n).fit().params[1])
    
    X_c = sm.add_constant(np.column_stack([D, C]))
    b_conC.append(sm.OLS(Y, X_c).fit().params[1])

print(f"Promedio b_noC: {np.mean(b_noC):.4f} (esperado ~0)")
print(f"Promedio b_conC: {np.mean(b_conC):.4f} (SESGADO)")

## PARTE 5: Visualización

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mediador
axes[0].hist(b_total, bins=30, alpha=0.5, label='Total (Y~D)', color='blue')
axes[0].hist(b_directo, bins=30, alpha=0.5, label='Directo (Y~D+M)', color='red')
axes[0].axvline(x=2, color='blue', linestyle='--', label='Efecto real = 2')
axes[0].axvline(x=0, color='red', linestyle='--')
axes[0].set_title('Mediador: Distribución del coef de D')
axes[0].legend()

# Colisionador
axes[1].hist(b_noC, bins=30, alpha=0.5, label='Sin C', color='green')
axes[1].hist(b_conC, bins=30, alpha=0.5, label='Con C', color='orange')
axes[1].axvline(x=0, color='black', linestyle='--', label='Efecto real = 0')
axes[1].set_title('Colisionador: Distribución del coef de D')
axes[1].legend()

plt.tight_layout()
plt.show()

## Resumen

| Tipo de variable | Estructura | ¿Controlar? | Consecuencia de controlar |
|------------------|------------|-------------|---------------------------|
| Mediador | D → M → Y | NO (para efecto total) | Bloquea ruta causal |
| Colisionador | D → C ← Y | NUNCA | Abre camino espurio |